# TTS 文本转语音教程

本教程介绍端到端的文本转语音 (Text-to-Speech) 系统，包括：

1. **TTS 系统架构** - 文本编码器 + 声学模型 + 声码器
2. **文本编码器** - 字符嵌入 + 卷积 + Transformer
3. **声学模型** - Tacotron 风格的 Mel 解码器
4. **HiFi-GAN 声码器** - 高保真波形生成
5. **损失函数与训练** - Mel 损失 + 停止 token 损失

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. TTS 系统架构

端到端 TTS 系统通常包含三个主要组件：

```
┌─────────────────────────────────────────────────────────────┐
│                    TTS 系统架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  文本 ──→ [文本编码器] ──→ 文本特征                          │
│                              ↓                               │
│                      [声学模型/解码器]                        │
│                              ↓                               │
│                        Mel 频谱图                            │
│                              ↓                               │
│                         [声码器]                             │
│                              ↓                               │
│                        音频波形                              │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from tts import TTSConfig, TextToSpeech, create_tts_model

# 查看默认配置
config = TTSConfig()
print("TTS 默认配置:")
print(f"  vocab_size: {config.vocab_size}")
print(f"  n_mels: {config.n_mels}")
print(f"  encoder_dim: {config.encoder_dim}")
print(f"  decoder_dim: {config.decoder_dim}")
print(f"  sample_rate: {config.sample_rate}")

In [ ]:
# 创建不同大小的模型
print("不同大小的 TTS 模型:")
for size in ["tiny", "base", "large"]:
    model = create_tts_model(size)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {size}: encoder_dim={model.config.encoder_dim}, params={n_params/1e6:.1f}M")

## 2. 文本编码器

文本编码器将输入文本转换为隐藏表示：

1. **字符嵌入**: 将字符 ID 转换为向量
2. **卷积层**: 提取局部特征
3. **Transformer 编码器**: 捕捉全局依赖

In [ ]:
from tts import TextEncoder, ConvBlock, PositionalEncoding

# 创建文本编码器
encoder_config = TTSConfig(
    vocab_size=256,
    encoder_dim=256,
    encoder_conv_layers=3,
    encoder_layers=4,
    encoder_heads=4
)
text_encoder = TextEncoder(encoder_config)

# 模拟输入文本 (字符 ID)
text = torch.randint(0, 256, (2, 30))  # [batch, seq_len]
encoder_output = text_encoder(text)

print(f"输入文本形状: {text.shape}")
print(f"编码器输出形状: {encoder_output.shape}")

In [ ]:
# 可视化卷积块
conv_block = ConvBlock(in_channels=256, out_channels=256, kernel_size=5)
x = torch.randn(2, 256, 30)  # [batch, channels, seq_len]
conv_output = conv_block(x)

print(f"卷积块输入形状: {x.shape}")
print(f"卷积块输出形状: {conv_output.shape}")
print("\n卷积块结构:")
print(conv_block)

## 3. 声学模型 (Mel 解码器)

声学模型将文本特征转换为 Mel 频谱图：

### 3.1 预网络 (Prenet)
预网络对解码器输入进行处理，增加模型的鲁棒性。

In [ ]:
from tts import Prenet, Postnet, MelDecoder

# 预网络
prenet = Prenet(in_dim=80, hidden_dim=256, out_dim=256)
mel_input = torch.randn(2, 50, 80)  # [batch, time, n_mels]
prenet_output = prenet(mel_input)

print(f"Prenet 输入形状: {mel_input.shape}")
print(f"Prenet 输出形状: {prenet_output.shape}")
print("\nPrenet 结构:")
print(prenet)

### 3.2 后网络 (Postnet)
后网络对 Mel 频谱进行精修，提高合成质量。

In [ ]:
# 后网络
postnet_config = TTSConfig(
    n_mels=80,
    postnet_channels=512,
    postnet_kernel=5,
    postnet_layers=5
)
postnet = Postnet(postnet_config)

mel = torch.randn(2, 80, 100)  # [batch, n_mels, time]
postnet_output = postnet(mel)

print(f"Postnet 输入形状: {mel.shape}")
print(f"Postnet 输出形状: {postnet_output.shape}")
print("\nPostnet 是残差连接: mel_refined = mel + postnet(mel)")

### 3.3 完整的 Mel 解码器

In [ ]:
# 创建 Mel 解码器
decoder_config = TTSConfig(
    n_mels=80,
    decoder_dim=256,
    decoder_heads=4,
    decoder_layers=4,
    prenet_dim=256,
    postnet_channels=512,
    postnet_layers=5
)
mel_decoder = MelDecoder(decoder_config)

# 训练模式: Teacher forcing
encoder_output = torch.randn(2, 30, 256)  # 编码器输出
mel_target = torch.randn(2, 80, 50)  # 目标 Mel 频谱

mel_output, mel_postnet, stop_tokens = mel_decoder(encoder_output, mel_target)

print(f"编码器输出形状: {encoder_output.shape}")
print(f"目标 Mel 形状: {mel_target.shape}")
print(f"\n解码器输出:")
print(f"  mel_output: {mel_output.shape}")
print(f"  mel_postnet: {mel_postnet.shape}")
print(f"  stop_tokens: {stop_tokens.shape}")

## 4. HiFi-GAN 声码器

HiFi-GAN 是一种高效的神经声码器，将 Mel 频谱转换为波形。

### 4.1 生成器架构

```
Mel 频谱 → 卷积 → [上采样 + 残差块] × N → 卷积 → tanh → 波形
```

In [ ]:
from tts import HiFiGANGenerator, ResBlock

# 创建 HiFi-GAN 生成器
vocoder_config = TTSConfig(
    n_mels=80,
    vocoder_upsample_rates=(8, 8, 2, 2),
    vocoder_upsample_kernel_sizes=(16, 16, 4, 4),
    vocoder_resblock_kernel_sizes=(3, 7, 11),
    vocoder_resblock_dilation_sizes=((1, 3, 5), (1, 3, 5), (1, 3, 5)),
    vocoder_initial_channel=512
)
vocoder = HiFiGANGenerator(vocoder_config)

# 计算上采样率
upsample_rate = np.prod(vocoder_config.vocoder_upsample_rates)
print(f"上采样率: {' × '.join(map(str, vocoder_config.vocoder_upsample_rates))} = {upsample_rate}")

In [ ]:
# 测试声码器
mel_input = torch.randn(2, 80, 10)  # [batch, n_mels, time]
waveform = vocoder(mel_input)

print(f"Mel 输入形状: {mel_input.shape}")
print(f"波形输出形状: {waveform.shape}")
print(f"  - 时间维度: {mel_input.shape[2]} × {upsample_rate} = {waveform.shape[2]}")
print(f"\n输出范围: [{waveform.min().item():.3f}, {waveform.max().item():.3f}] (tanh 激活)")

### 4.2 残差块

In [ ]:
# 残差块
resblock = ResBlock(channels=256, kernel_size=3, dilations=(1, 3, 5))
x = torch.randn(2, 256, 100)
resblock_output = resblock(x)

print(f"残差块输入形状: {x.shape}")
print(f"残差块输出形状: {resblock_output.shape}")
print("\n残差块使用不同的膨胀率来捕捉不同尺度的特征")
print(f"膨胀率: {(1, 3, 5)}")

## 5. 完整 TTS 模型

In [ ]:
# 创建完整的 TTS 模型
model_config = TTSConfig(
    vocab_size=256,
    n_mels=80,
    encoder_dim=256,
    encoder_layers=4,
    decoder_dim=256,
    decoder_layers=4,
    vocoder_initial_channel=256
)
tts_model = TextToSpeech(model_config)

# 模拟输入
text = torch.randint(0, 256, (2, 20))  # [batch, text_len]
mel_target = torch.randn(2, 80, 50)  # [batch, n_mels, mel_len]

# 训练前向传播
output = tts_model(text, mel_target)

print(f"输入文本形状: {text.shape}")
print(f"目标 Mel 形状: {mel_target.shape}")
print(f"\n输出:")
print(f"  mel_output: {output['mel_output'].shape}")
print(f"  mel_postnet: {output['mel_postnet'].shape}")
print(f"  stop_tokens: {output['stop_tokens'].shape}")

## 6. 损失函数

In [ ]:
from tts import tts_loss

# 准备数据
mel_output = output['mel_output']
mel_postnet = output['mel_postnet']
stop_tokens = output['stop_tokens']

# 创建停止 token 目标 (最后几帧为 1)
stop_target = torch.zeros(2, 50)
stop_target[:, -5:] = 1.0

# 计算损失
total_loss, loss_dict = tts_loss(
    mel_output, mel_postnet, mel_target,
    stop_tokens, stop_target
)

print("TTS 损失函数:")
print(f"  Mel 损失: {loss_dict['mel_loss'].item():.4f}")
print(f"  Mel Postnet 损失: {loss_dict['mel_postnet_loss'].item():.4f}")
print(f"  停止 token 损失: {loss_dict['stop_loss'].item():.4f}")
print(f"  总损失: {total_loss.item():.4f}")

## 7. 语音合成推理

In [ ]:
# 推理模式
tts_model.eval()

# 合成语音
with torch.no_grad():
    waveform = tts_model.synthesize(text)

print(f"输入文本形状: {text.shape}")
print(f"合成波形形状: {waveform.shape}")
print(f"\n波形范围: [{waveform.min().item():.3f}, {waveform.max().item():.3f}]")

In [ ]:
# 可视化合成的波形
waveform_np = waveform[0, 0].numpy()

plt.figure(figsize=(14, 4))
plt.plot(waveform_np[:2000])  # 显示前 2000 个采样点
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.title('Synthesized Waveform (first 2000 samples)')
plt.tight_layout()
plt.show()

## 8. 模型参数统计

In [ ]:
def model_summary(model):
    """打印模型摘要"""
    print("模型组件参数统计:")
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        print(f"  {name}: {params/1e6:.2f}M")
    
    total = sum(p.numel() for p in model.parameters())
    print(f"\n总参数: {total/1e6:.2f}M")

model_summary(tts_model)

## 9. 总结

本教程介绍了 TTS 文本转语音系统的核心组件：

| 组件 | 功能 | 关键特性 |
|:-----|:-----|:---------|
| 文本编码器 | 提取文本特征 | 嵌入 + 卷积 + Transformer |
| Mel 解码器 | 生成 Mel 频谱 | Prenet + Transformer + Postnet |
| HiFi-GAN | 波形生成 | 上采样 + 多尺度残差块 |

**TTS 系统的关键点**:
- Prenet 增加鲁棒性
- Postnet 精修 Mel 频谱
- 停止 token 控制生成长度
- HiFi-GAN 实现高保真波形合成

**TTS 模型对比**:

| 模型 | 类型 | 速度 | 质量 |
|:-----|:-----|:-----|:-----|
| Tacotron 2 | 自回归 | 慢 | 高 |
| FastSpeech 2 | 非自回归 | 快 | 高 |
| VITS | 端到端 | 快 | 很高 |